<a href="https://colab.research.google.com/github/tomasbelak24/deeplearning-vision/blob/main/hw1/best_model_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p models
# I might make the repo private in the future so the files from /baseline directory from the zip file might have to be used
!wget -O models/030.pth https://raw.githubusercontent.com/tomasbelak24/deeplearning-vision/main/hw1/models/030.pth # minimal val loss
!wget -O models/096.pth https://raw.githubusercontent.com/tomasbelak24/deeplearning-vision/main/hw1/models/096.pth # maximal val acc
!wget -O models/baseline.pth https://raw.githubusercontent.com/tomasbelak24/deeplearning-vision/main/hw1/models/baseline_model.pth # baseline


--2025-11-20 13:18:51--  https://raw.githubusercontent.com/tomasbelak24/deeplearning-vision/main/hw1/models/030.pth
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1986870 (1.9M) [application/octet-stream]
Saving to: ‘models/030.pth’

models/030.pth      100%[===================>]   1.89M  --.-KB/s    in 0.03s   

2025-11-20 13:18:51 (54.3 MB/s) - ‘models/030.pth’ saved [1986870/1986870]

--2025-11-20 13:18:51--  https://raw.githubusercontent.com/tomasbelak24/deeplearning-vision/main/hw1/models/096.pth
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting respons

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

import numpy as np
from matplotlib import pyplot as plt

In [ ]:
# @title Loading CIFAR10 dataset

#reused code from the 5th lab

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

transform = transforms.ToTensor()

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:03<00:00, 47.8MB/s]


In [ ]:
from torch.nn import ReLU, Softmax, Sequential, Linear, Conv2d, MaxPool2d, Flatten, AdaptiveAvgPool2d, Dropout, Dropout2d, BatchNorm2d, CrossEntropyLoss

ce_loss = CrossEntropyLoss()

def build_deep_model(path=None, num_classes=10):
    model = Sequential(
        Conv2d(3, 32, 3, padding=1, bias=True),
        BatchNorm2d(32),
        ReLU(),
        Conv2d(32, 32, 3, padding=1, bias=True),
        BatchNorm2d(32),
        ReLU(),
        Conv2d(32, 32, 3, padding=1, bias=True),
        BatchNorm2d(32),
        ReLU(),
        MaxPool2d(2),

        Conv2d(32, 64, 3, padding=1, bias=True),
        BatchNorm2d(64),
        ReLU(),
        Conv2d(64, 64, 3, padding=1, bias=True),
        BatchNorm2d(64),
        ReLU(),
        Conv2d(64, 64, 3, padding=1, bias=True),
        BatchNorm2d(64),
        ReLU(),
        MaxPool2d(2),

        Conv2d(64, 128, 3, padding=1, bias=True),
        BatchNorm2d(128),
        ReLU(),
        Conv2d(128, 128, 3, padding=1, bias=True),
        BatchNorm2d(128),
        ReLU(),
        Conv2d(128, 128, 3, padding=1, bias=True),
        BatchNorm2d(128),
        ReLU(),
        MaxPool2d(2),

        AdaptiveAvgPool2d(1),
        Flatten(),
        Linear(128, 64),
        ReLU(),
        Linear(64, num_classes)
    )
    if path is not None:
      model.load_state_dict(torch.load(path))

    model_inference = Sequential(model, Softmax(dim=1))
    return model, model_inference


def build_baseline_model(path=None, num_classes=10):
    model = Sequential(
          Conv2d(3, 32, 3, padding=1, bias=True), ReLU(),
          MaxPool2d(2),
          Conv2d(32, 64, 3, padding=1, bias=True), ReLU(),
          MaxPool2d(2),
          Conv2d(64, 128, 3, padding=1, bias=True), ReLU(),
          MaxPool2d(2),

          AdaptiveAvgPool2d(1),
          Flatten(),
          Linear(128, num_classes)
    )

    if path is not None:
        model.load_state_dict(torch.load(path))

    model_inference = Sequential(model, Softmax(dim=1))
    return model, model_inference

In [ ]:
# @title Baseline evaluation
batch_size=32
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model, model_inference = build_baseline_model('models/baseline.pth')
model.to(device)

dataloader_test = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)
model.eval()

with torch.no_grad():
  correct = 0
  total = 0
  for i, batch in enumerate(dataloader_test):
    x, y = batch[0].to(device), batch[1].to(device)

    out = model(x)
    loss = ce_loss(out, y)
    acc = torch.sum(torch.argmax(out, dim=-1) == y)
    correct += acc.item()
    total += len(batch[1])

acc = correct / total
print("Test set accuracy: ", acc)



cuda:0
Test set accuracy:  0.7206


In [ ]:
# @title Best val loss model evaluation
batch_size=64
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model, model_inference = build_deep_model('models/030.pth')
model.to(device)

dataloader_test = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)
model.eval()

with torch.no_grad():
  correct = 0
  total = 0
  for i, batch in enumerate(dataloader_test):
    x, y = batch[0].to(device), batch[1].to(device)

    out = model(x)
    loss = ce_loss(out, y)
    acc = torch.sum(torch.argmax(out, dim=-1) == y)
    correct += acc.item()
    total += len(batch[1])

acc = correct / total
print("Test set accuracy: ", acc)



Test set accuracy:  0.8841


In [ ]:
# @title Best validation accuracy model evaluation
batch_size=64
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model, model_inference = build_deep_model('models/096.pth')
model.to(device)

dataloader_test = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)
model.eval()

with torch.no_grad():
  correct = 0
  total = 0
  for i, batch in enumerate(dataloader_test):
    x, y = batch[0].to(device), batch[1].to(device)

    out = model(x)
    loss = ce_loss(out, y)
    acc = torch.sum(torch.argmax(out, dim=-1) == y)
    correct += acc.item()
    total += len(batch[1])

acc = correct / total
print("Test set accuracy: ", acc)



Test set accuracy:  0.8926
